# KB AADS — Containment agent RL training (Colab)

Trains the RLlib PPO policy for `kb-aads`'s Containment agent — given a process's real syscall-behavior-derived state, decide a `ContainmentLevel` (`NONE`/`CGROUP`/`SECCOMP`/`NAMESPACE`/`TERMINATE`).

**Data provenance, stated plainly:** training data comes from real Linux syscall traces (ADFA-LD, Creech & Hu, UNSW Canberra) mapped onto KB's actual attack-lab scenario names, plus real `/proc` telemetry sampled from the dev machine for the benign class — see `scripts/dataset/collect.py` and `label.py` in the repo for exactly how. This is a scoped-down stand-in for the project's full Phase 0 pipeline (real eBPF capture from isolated-VM attack-lab runs), not that pipeline itself — see `docs/development/control-aads/aads-intelligence-roadmap.md`.

**Before running:** if you unzipped the training bundle directly into your Colab session (so `data/train.csv`, `data/val.csv`, `data/test.csv` already exist), the cell below will detect that and skip the upload prompt automatically. Otherwise it'll ask you to upload the three CSVs from `scripts/dataset/output/`.

In [ ]:
!pip install -q "ray[rllib]>=2.9.0" gymnasium torch pandas

In [ ]:
import os

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

_already_present = all(
    os.path.exists(os.path.join(DATA_DIR, f"{split}.csv")) for split in ("train", "val", "test")
)

if _already_present:
    print("train.csv/val.csv/test.csv already present in ./data/ — skipping upload.")
else:
    try:
        from google.colab import files
        print("Upload train.csv, val.csv, test.csv (from scripts/dataset/output/ in the repo):")
        uploaded = files.upload()
        for fname in uploaded:
            os.replace(fname, os.path.join(DATA_DIR, fname))
    except ImportError:
        print("Not running on Colab — expecting train.csv/val.csv/test.csv already in ./data/")

for split in ("train", "val", "test"):
    path = os.path.join(DATA_DIR, f"{split}.csv")
    assert os.path.exists(path), f"missing {path} — upload it before continuing"
print("data files present:", os.listdir(DATA_DIR))

## Environment

Mirrors `kb-aads/marl/env.py` in the repo — keep the two in sync if either changes. Observation is the real `ProcessState`/`KBEvent` wire-field shape (`score`, `zone`, `uid_is_root`, `score_delta`, `event_type`); action is `ContainmentLevel` (5 discrete values); single-step episodes (see `env.py`'s docstring for why that's an honest scope choice for Containment specifically, not a corner cut).

In [ ]:
import random

import gymnasium as gym
import numpy as np
import pandas as pd
from gymnasium import spaces

NONE, CGROUP, SECCOMP, NAMESPACE, TERMINATE = 0, 1, 2, 3, 4
N_CONTAINMENT_LEVELS = 5
LEVEL_NAMES = ["NONE", "CGROUP", "SECCOMP", "NAMESPACE", "TERMINATE"]


class ContainmentEnv(gym.Env):
    def __init__(self, env_config=None):
        super().__init__()
        env_config = env_config or {}
        csv_path = env_config["csv_path"]
        df = pd.read_csv(csv_path)
        self._rows_by_category = {
            cat: sub.to_dict("records") for cat, sub in df.groupby("category")
        }
        self._categories = list(self._rows_by_category.keys())

        self.observation_space = spaces.Box(
            low=np.array([0.0, 0.0, 0.0, -100.0, 0.0], dtype=np.float32),
            high=np.array([100.0, 2.0, 1.0, 100.0, 3.0], dtype=np.float32),
            dtype=np.float32,
        )
        self.action_space = spaces.Discrete(N_CONTAINMENT_LEVELS)
        self._current_target = None

    def _sample_row(self):
        cat = random.choice(self._categories)
        return random.choice(self._rows_by_category[cat])

    def _obs_from_row(self, row):
        return np.array([
            row["score"], row["zone"], row["uid_is_root"],
            row["score_delta"], row["event_type"],
        ], dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        row = self._sample_row()
        self._current_target = int(row["target_containment"])
        return self._obs_from_row(row), {"target_containment": self._current_target}

    def _reward(self, action, target):
        if action == target:
            return 1.0
        distance = abs(action - target)
        penalty = 0.3 * distance
        if action < target:
            penalty += 0.2 * distance
        return round(1.0 - penalty, 4)

    def step(self, action):
        reward = self._reward(int(action), self._current_target)
        obs = np.zeros(5, dtype=np.float32)
        return obs, reward, True, False, {"target_containment": self._current_target}

## Train

In [ ]:
import ray
from ray.rllib.algorithms.ppo import PPOConfig

ray.init(ignore_reinit_error=True)

config = (
    PPOConfig()
    .environment(ContainmentEnv, env_config={"csv_path": os.path.join(DATA_DIR, "train.csv")})
    .framework("torch")
    .env_runners(num_env_runners=1)
    .training(lr=5e-4, train_batch_size=2000, minibatch_size=256, num_epochs=10)
)
algo = config.build()
print("RLlib PPO algorithm built.")

In [ ]:
import torch


def evaluate(algo, csv_path, n_samples=2000):
    df = pd.read_csv(csv_path)
    if len(df) > n_samples:
        df = df.sample(n_samples, random_state=0)

    module = algo.get_module()
    correct = 0
    per_class_total = {name: 0 for name in LEVEL_NAMES}
    per_class_correct = {name: 0 for name in LEVEL_NAMES}
    under_contain = 0

    for _, row in df.iterrows():
        obs = np.array([row["score"], row["zone"], row["uid_is_root"],
                         row["score_delta"], row["event_type"]], dtype=np.float32)
        target = int(row["target_containment"])
        with torch.no_grad():
            out = module.forward_inference({"obs": torch.from_numpy(obs).unsqueeze(0)})
            action = int(torch.argmax(out["action_dist_inputs"], dim=-1).item())
        per_class_total[LEVEL_NAMES[target]] += 1
        if action == target:
            correct += 1
            per_class_correct[LEVEL_NAMES[target]] += 1
        elif action < target:
            under_contain += 1

    n = len(df)
    print(f"overall accuracy: {correct}/{n} ({100*correct/n:.1f}%)")
    print(f"under-containment rate: {under_contain}/{n} ({100*under_contain/n:.1f}%)")
    for name in LEVEL_NAMES:
        total = per_class_total[name]
        if total:
            print(f"  {name:10s}: {per_class_correct[name]}/{total} ({100*per_class_correct[name]/total:.1f}%)")


print("Baseline (untrained) policy on val set:")
evaluate(algo, os.path.join(DATA_DIR, "val.csv"))

In [ ]:
ITERATIONS = 60

for i in range(ITERATIONS):
    result = algo.train()
    if (i + 1) % 5 == 0 or i == 0:
        r = result.get("env_runners", {}).get("episode_return_mean", float("nan"))
        print(f"iter {i+1}/{ITERATIONS}  episode_return_mean={r:.3f}")

print("\nTrained policy on val set:")
evaluate(algo, os.path.join(DATA_DIR, "val.csv"))

In [ ]:
print("Held-out test set (final report-out number):")
evaluate(algo, os.path.join(DATA_DIR, "test.csv"))

## Save + download the checkpoint

Download the zip and extract it into `kb-aads/marl/checkpoints/containment_ppo/` in the repo — `agents/containment.py` loads it from there at swarm start.

In [ ]:
import shutil

CHECKPOINT_DIR = "containment_ppo"
algo.save(CHECKPOINT_DIR)
shutil.make_archive("containment_ppo_checkpoint", "zip", CHECKPOINT_DIR)
print("Saved: containment_ppo_checkpoint.zip")

try:
    from google.colab import files
    files.download("containment_ppo_checkpoint.zip")
except ImportError:
    print(f"Not on Colab — checkpoint zip is at {os.path.abspath('containment_ppo_checkpoint.zip')}")